In [ ]:
!pip install scikit-learn transformers pycountry requests

import os
import json
import numpy as np
import colorsys
import torch
import tensorflow as tf
import base64
import requests
import io
import textwrap

from PIL import Image
from torchvision import transforms
from sklearn.cluster import KMeans
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from google.colab import drive
from google.colab import files
from IPython.display import display

# Mount Drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 51.8 MB/s eta 0:00:00
Mounted at /content/drive


In [ ]:
# ==========================================
# 1. LOAD VIT EMOTION MODEL
# ==========================================
vit_model_path = "/content/drive/MyDrive/fypModels/HiddenTales_Model/best.torchscript"
vit_classes_path = "/content/drive/MyDrive/fypModels/HiddenTales_Model/classes.json"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

emotion_model = torch.jit.load(vit_model_path, map_location=device)
emotion_model.eval()

with open(vit_classes_path, "r") as f:
    emotion_classes = json.load(f)

# Define Transform
vit_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])
print("✅ ViT Emotion Model loaded.")

# ==========================================
# 2. LOAD COLOUR MODEL
# ==========================================
color_model_path = "/content/drive/MyDrive/fypModels/color_model_export/color_model_export/HiddenTales_ColorModel_Improved.keras"
color_label_path = "/content/drive/MyDrive/fypModels/color_model_export/color_model_export/label_names.json"

color_model = tf.keras.models.load_model(color_model_path)
with open(color_label_path, "r") as f:
    color_label_names = json.load(f)

# specific fix for shape issue
try:
    input_shape = color_model.input_shape[1:3]
    if input_shape[0] is None:
        input_shape = (28, 28)
except Exception:
    input_shape = (28, 28)

print(f"✅ Colour Model loaded. Input size: {input_shape}")

# ==========================================
# 3. LOAD PSYCHOLOGY T5 MODEL
# ==========================================
psych_model_path = "/content/drive/MyDrive/fypModels/nlp_model_export/nlp_model_export/HiddenTales_T5_Final"

psych_tokenizer = AutoTokenizer.from_pretrained(psych_model_path, local_files_only=True)
psych_model = AutoModelForSeq2SeqLM.from_pretrained(psych_model_path, local_files_only=True).to(device)

print("✅ Psychology T5 Model loaded.")

✅ ViT Emotion Model loaded.
✅ Colour Model loaded. Input size: (32, 32)
✅ Psychology T5 Model loaded.


In [ ]:
# ==========================================
# HELPER FUNCTIONS FOR COLOR ANALYSIS
# ==========================================

def remove_background(img):
    """Rough white/background removal based on corner colours."""
    arr = np.array(img)
    if len(arr.shape) < 3:
        return arr

    h, w, _ = arr.shape
    # Sample corners
    corners = [
        arr[0:30, 0:30],
        arr[0:30, w-30:w],
        arr[h-30:h, 0:30],
        arr[h-30:h, w-30:w],
    ]
    corner_pixels = np.vstack([c.reshape(-1, 3) for c in corners])
    bg_color = np.median(corner_pixels, axis=0)

    # Calculate difference from background
    diff = np.sqrt(np.sum((arr - bg_color) ** 2, axis=2))
    mask = diff > 40
    result = arr[mask]

    if len(result) < 100:
        return arr[diff > 15] # be more lenient if image is empty

    return result

def get_saturation(rgb):
    r, g, b = rgb[0] / 255.0, rgb[1] / 255.0, rgb[2] / 255.0
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    return s, l

def predict_color(rgb):
    """Use CNN + saturation/lightness rules to get a colour name."""
    saturation, lightness = get_saturation(rgb)

    # Handle greys / black explicitly
    if lightness < 0.12: return "black"
    if saturation < 0.20:
        if lightness < 0.50: return "black"
        return "light grey"

    # Prepare for CNN
    h_in, w_in = input_shape
    patch = np.zeros((1, h_in, w_in, 3), dtype=np.uint8)
    patch[:] = rgb
    img_input = patch.astype("float32") / 255.0

    pred = color_model.predict(img_input, verbose=0)
    idx = np.argmax(pred)
    predicted_name = color_label_names[idx]

    # Merge nuances
    if predicted_name in ['grey', 'dark grey', 'charcoal', 'white', 'silver', 'beige']:
        if lightness < 0.40: return "black"
        return "light grey"

    return predicted_name

def get_dominant_colors(img, n_colors=5):
    """KMeans over foreground pixels to find top N RGB colours."""
    arr = remove_background(img)
    if len(arr) < 50:
        return []

    kmeans = KMeans(n_clusters=n_colors, random_state=42, n_init=10).fit(arr)
    centers = kmeans.cluster_centers_.astype(int)

    unique, counts = np.unique(kmeans.labels_, return_counts=True)
    scores = []

    # Weigh colors by count AND saturation (prefer colorful parts)
    for i, center in enumerate(centers):
        sat, _ = get_saturation(center)
        score = counts[i] * (1 + (sat * 2.5))
        scores.append(score)

    sorted_indices = np.argsort(scores)[::-1]
    return centers[sorted_indices]

In [ ]:
# ==========================================
# OPENROUTER CONFIGURATION
# ==========================================
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "your-openrouter-api-key")
OPENROUTER_MODEL = "qwen/qwen-2-vl-72b-instruct" # use Qwen or Llama vision models
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

def get_image_description_openrouter(img: Image.Image) -> str:
    """Send image to OpenRouter Vision Model for description."""

    # 1. Convert Image to Base64
    buf = io.BytesIO()
    img.save(buf, format="JPEG")
    img_bytes = buf.getvalue()
    img_b64 = base64.b64encode(img_bytes).decode("utf-8")
    img_data_url = f"data:image/jpeg;base64,{img_b64}"

    # 2. Prepare Payload
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://colab.research.google.com",
        "X-Title": "HiddenTales FYP"
    }

    prompt_text = (
        "You are describing children's drawings. "
        "Look at this image and write ONE clear paragraph (4-6 sentences) "
        "describing what is drawn: main objects, characters, setting, and "
        "overall atmosphere. Do NOT add psychological interpretation yet."
    )

    payload = {
        "model": OPENROUTER_MODEL,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt_text},
                    {"type": "image_url", "image_url": {"url": img_data_url}}
                ]
            }
        ],
        "temperature": 0.4,
        "max_tokens": 300
    }

    # 3. Send Request
    try:
        resp = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=90)
        resp.raise_for_status()
        data = resp.json()
        desc = data["choices"][0]["message"]["content"].strip()
        return desc
    except Exception as e:
        print(f"[OpenRouter ERROR]: {e}")
        return "Description unavailable due to API error."

In [ ]:
def analyze_drawing(img):
    # --- 1. Emotion (ViT) ---
    img_vit = vit_transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = emotion_model(img_vit)
        emotion_idx = torch.argmax(logits).item()
    emotion = emotion_classes[emotion_idx]

    # --- 2. Colour Analysis ---
    # This was the part failing before because get_dominant_colors was missing
    dom_colors = get_dominant_colors(img)

    if len(dom_colors) == 0:
        unique_colors = ["light grey"]
        primary_color = "light grey"
    else:
        all_preds = [predict_color(c) for c in dom_colors]
        unique_colors = []
        for c in all_preds:
            if c not in unique_colors:
                unique_colors.append(c)

        # Filter background colors
        real_colors = [c for c in unique_colors if c not in ["light grey", "white", "beige"]]
        if real_colors:
            unique_colors = real_colors

        primary_color = unique_colors[0] if unique_colors else "light grey"

    # --- 3. Psychology Report (T5) ---
    emotion_for_t5 = emotion.lower()

    # T5 Generation Logic
    psych_colors = unique_colors[:3] if unique_colors else ["light grey"]
    color_string = ", ".join(psych_colors)

    prompt = f"emotion: {emotion_for_t5} | colors: {color_string} | primary: {primary_color}"
    inputs = psych_tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = psych_model.generate(
            **inputs,
            max_length=220,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3,
            repetition_penalty=1.1,
            do_sample=False
        )
    psychology_report = psych_tokenizer.decode(output_ids[0], skip_special_tokens=True)

    return {
        "emotion": emotion,
        "colors": unique_colors,
        "primary_color": primary_color,
        "psychology_report": psychology_report
    }

def pretty_print_block(text, width=90):
    wrapper = textwrap.TextWrapper(width=width)
    for line in wrapper.wrap(text):
        print(line)

In [ ]:
print("\n=============================")
print(" READY! Upload a child's drawing")
print("=============================")

uploaded = files.upload()

if uploaded:
    img_name = list(uploaded.keys())[0]
    img = Image.open(img_name).convert("RGB")

    # 1. Run Internal Models (ViT, Color, T5)
    result = analyze_drawing(img)

    # 2. Run External ELLM API (OpenRouter)
    print("... Contacting OpenRouter Vision Model (ELLM) ...")
    ellm_description = get_image_description_openrouter(img)

    # 3. Final Report
    print("\n" + "="*60)
    print(" HIDDENTALES FULL REPORT")
    print("="*60)
    print(f" Emotion Detected: {result['emotion']}")
    print(f" Colors: {', '.join(result['colors'])}")
    print(f" Primary Color: {result['primary_color']}")
    print("-" * 20)
    print(" Image Description (via ELLM API):")
    pretty_print_block(ellm_description)
    print("-" * 20)
    print(" Psychology Report (T5 + Color Theory):")
    pretty_print_block(result['psychology_report'])
    print("="*60)

    display(img.resize((300, 300)))